In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
# import importlib

import utils as uti
# importlib.reload(uti)




In [ ]:
print(uti.__file__)
print(dir(uti))

# import data

In [ ]:
df = pd.read_pickle("../data/00_df_train.pkl")
df_test = pd.read_pickle("../data/00_df_test.pkl")

In [ ]:
print(df.shape)
print(df_test.shape)

In [ ]:
df.head()

In [ ]:
df.groupby('bad').size()

In [ ]:
df.info()

# correlation pre transform

In [ ]:
correlation_matrix = df.corr(numeric_only=True)

plt.figure(figsize=(10, 10))
sns.heatmap(
    correlation_matrix,
    # annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

plt.show()

In [ ]:
sns.pairplot(
    df.sample(n=250, random_state=42).reset_index(drop=True),
    hue="bad",
    height=2.5,
    diag_kind="kde",
    plot_kws={"alpha": 0.6}
)

plt.show()

# domographics

In [ ]:
df.columns

In [ ]:
uti.plot_event_num(
    df=df,
    column_names=["age_years"],
    event_column="bad",
    n_bins=10
)

In [ ]:
uti.plot_event_cat(
    df,
    ['personal_status_sex'],
    event_column="bad"
)

# weight of evidence

In [ ]:
df.columns

In [ ]:
num_cols = df.select_dtypes(include='number').columns.tolist()
excl = ['bad','age_years']

num_cols = [item for item in num_cols if item not in excl]
num_cols

In [ ]:
cat_cols = df.select_dtypes(exclude='number').columns.tolist()
excl = ['personal_status_sex']

cat_cols = [item for item in cat_cols if item not in excl]
cat_cols

In [ ]:
import toad

combiner = toad.transform.Combiner()

combiner.fit(
    df[num_cols + cat_cols],
    df['bad'],
    method="chi",
    min_samples=0.05
)

In [ ]:
X_train_bins = combiner.transform(df[num_cols + cat_cols])
X_test_bins = combiner.transform(df_test[num_cols + cat_cols])

In [ ]:
X_train_bins.head()

In [ ]:
bin_rules = combiner.export()

print(bin_rules)

In [ ]:
woe_transformer = toad.transform.WOETransformer()

woe_transformer.fit(
    X_train_bins,
    df['bad']
)


In [ ]:
X_train_woe = woe_transformer.transform(X_train_bins)
X_test_woe = woe_transformer.transform(X_test_bins)

In [ ]:
X_train_woe.head()

## check bins

In [ ]:
# Count the number of final bins for each variable
bin_counts = X_train_woe.nunique(dropna=False)
bin_counts

In [ ]:
valid_features = bin_counts[bin_counts > 1].index.tolist()
dropped_features = bin_counts[bin_counts <= 1].index.tolist()

print("Keeping:", valid_features)
print("Dropping:", dropped_features)


In [ ]:
X_train_woe.drop(columns=dropped_features, inplace=True)
X_test_woe.drop(columns=dropped_features, inplace=True)

In [ ]:
X_train_woe = X_train_woe.add_suffix("_woe")
X_test_woe = X_test_woe.add_suffix("_woe")

In [ ]:
vars_woe = X_train_woe.columns.tolist()

In [ ]:
print(X_train_woe.shape)
print(X_test_woe.shape)

In [ ]:
print(df.shape)
print(df_test.shape)

In [ ]:
df = pd.concat(
    [df, X_train_woe],
    axis=1
)

df_test = pd.concat(
    [df_test, X_test_woe],
    axis=1
)


In [ ]:
print(df.shape)
print(df_test.shape)

In [ ]:
df.head()

In [ ]:
uti.plot_event_cat(
    pd.concat([df[vars_woe].round(4).astype(object),df['bad']], axis=1),
    vars_woe,
    event_column="bad"
)

## save feature names

In [ ]:
feat_names = {}

In [ ]:
feat_names['raw'] = [num_cols + cat_cols]
feat_names['woe'] = vars_woe

In [ ]:
feat_names

In [ ]:
import pickle

with open("../data/01_dic_feat_names.pkl", "wb") as file:
    pickle.dump(feat_names, file)


# with open("my_dict.pkl", "rb") as file:
#     my_dict = pickle.load(file)

## save woe data

In [ ]:
print(df.shape)
print(df_test.shape)

In [ ]:
df.to_pickle("../data/01_df_train.pkl")
df_test.to_pickle("../data/01_df_test.pkl")


# univariate variable importance

In [ ]:
df.info()

In [ ]:
walds = uti.calculate_wald_statistics(
    df=df,
    column_names=feat_names['woe'],
    event_column="bad"
)

In [ ]:
walds

# correlation by importance

In [ ]:
plt.figure(figsize=(10, 10))
sns.heatmap(
    df[["bad"] + feat_names['woe']].corr(),
    # annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

plt.show()

In [ ]:
tmp = uti.corr_crit(correlation_matrix,.3)

In [ ]:
walds = pd.merge(walds, tmp, on="var", how="left")

In [ ]:
walds

## correlation with criteria

In [ ]:
plt.figure(figsize=(10, 10))
sns.heatmap(
    df[walds[walds['corr ind'] == False]['var'].tolist() + ['bad']].corr(),
    # annot=True,
    cmap="coolwarm",
    vmin=-1,
    vmax=1
)

plt.show()

# save wald table

In [ ]:
walds.to_pickle("../data/01_df_walds.pkl")